# Lecture 10 — Functions, Tables, and Sampling

**부산대학교 데이터과학입문 (2026-1)** · 최동희 (`dchoi@pusan.ac.kr`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongheechoi/2026_ds_example/blob/main/notebooks/10_functions_tables_sampling.ipynb)

이 노트북은 강의 슬라이드(`10_functions_tables_sampling.tex`)의 모든 코드 예제를 Google Colab에서 바로 실행할 수 있도록 구성한 것입니다.

데이터는 GitHub raw URL에서 직접 읽어오므로 별도 파일 업로드가 필요하지 않습니다.

## 목차
**Part A — Functions and Tables**
1. One-line histogram with Pandas
2. User-defined Functions (`def`)
3. `apply()` — function over a column
4. `groupby()` — aggregation
5. `merge` — joining tables

**Part B — Sampling and Empirical Distribution**

6. Deterministic vs Random sample
7. Empirical Distribution and Law of Large Numbers
8. Parameter vs Statistic
9. Simulating a sampling distribution

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')

# Data is hosted in this repo. Reading directly via raw URLs (no upload needed).
DATA_BASE = 'https://raw.githubusercontent.com/dongheechoi/2026_ds_example/main/data'

---
## 1. One-line histogram with Pandas

In [ ]:
top = pd.read_csv(f'{DATA_BASE}/top_movies_2017.csv')
top.head()

In [ ]:
top['Year'].plot.hist(bins=20, edgecolor='white')
plt.xlabel('Year')
plt.title('Top 200 movies by year')
plt.show()

Same plot via **seaborn** (used in Topic 9):

In [ ]:
import seaborn as sns
sns.set_theme(style='darkgrid')

fig, ax = plt.subplots()
sns.histplot(data=top, x='Year', bins=20, ax=ax)
ax.set_title('Top 200 movies by year (sns.histplot)')
plt.show()

**Both APIs above produce the same plot** — `df.plot.hist()` is the Pandas-native shortcut, `sns.histplot()` is the seaborn version we used in Topic 9. Either is fine.

*Reminder:* a **bar** chart visualizes a categorical distribution; a **histogram** visualizes a numerical one. With unequal bin widths, area (not height) represents the proportion.

---
## 2. User-defined Functions (`def`)

In [ ]:
def spread(values):
    """Difference between the largest and smallest value."""
    return max(values) - min(values)

spread([3, 7, 2, 9, 5])

### Discussion — what does this function do?

In [ ]:
def f(s):
    return np.round(np.array(s) / sum(s) * 100, 2)

f([10, 30, 60])

---
## 3. `apply()` — function over a column

In [ ]:
def cut_off_at_100(x):
    """The smaller of x and 100."""
    return min(x, 100)

ages = pd.DataFrame({
    'Person': ['A', 'B', 'C', 'D', 'E', 'F'],
    'Age':    [17, 117, 52, 100, 6, 101]
})
ages['Cut Off Age'] = ages['Age'].apply(cut_off_at_100)
ages

### Predicting child height with `apply()` (Galton family data)

In [ ]:
family = pd.read_csv(f'{DATA_BASE}/galton.csv')
family.head()

In [ ]:
parent_avg = (family['father'] + family['mother']) / 2
heights = pd.DataFrame({
    'Parent Average': parent_avg,
    'Child': family['childHeight']
})

def predict_child(p_avg):
    nearby = heights[(p_avg - 0.5 <= heights['Parent Average']) &
                     (heights['Parent Average'] < p_avg + 0.5)]
    return np.mean(nearby['Child'])

heights['Prediction'] = heights['Parent Average'].apply(predict_child)
heights.head()

In [ ]:
fig, ax = plt.subplots()
heights.plot.scatter(x='Parent Average', y='Child', alpha=0.4, ax=ax)
heights.sort_values('Parent Average').plot.line(
    x='Parent Average', y='Prediction', color='red', ax=ax)
ax.set_title('Average parent height vs child height (with apply prediction)')
plt.show()

---
## 4. `groupby()` — aggregation

In [ ]:
cones = pd.DataFrame({
    'Flavor': ['strawberry', 'chocolate', 'chocolate', 'strawberry', 'chocolate'],
    'Price':  [3.55, 4.75, 6.55, 5.25, 5.25]
})
cones

In [ ]:
# (1) Number of rows per group
cones.groupby('Flavor').size()

In [ ]:
# (2) Mean price per group
cones.groupby('Flavor')['Price'].mean()

### Grouping by multiple columns

In [ ]:
more_cones = pd.DataFrame({
    'Flavor': ['strawberry','chocolate','chocolate','strawberry','chocolate','bubblegum'],
    'Color':  ['pink','light brown','dark brown','pink','dark brown','pink'],
    'Price':  [3.55, 4.75, 5.25, 5.25, 5.25, 4.75]
})

more_cones.groupby(['Flavor', 'Color']).size().reset_index(name='count')

---
## 5. `merge` — joining two tables

### Setup: two small tables

`sales` lists how many of each product was sold; `prices` is a price list. Note: ProductID 4 is in `sales` but not in `prices`, and ProductID 5 is the opposite.

In [ ]:
sales = pd.DataFrame({
    'ProductID': [1, 2, 3, 4],
    'Sold':      [10, 25, 7, 15]
})
prices = pd.DataFrame({
    'ProductID': [1, 2, 3, 5],
    'Price':     [3.5, 4.0, 5.0, 2.5]
})
display(sales); display(prices)

### (a) `inner` — keys present in **both** (default)

In [ ]:
sales.merge(prices, on='ProductID')   # how='inner' is the default

IDs 4 (no price) and 5 (no sale) are dropped.

### (b) `left` — all rows of the **left** table

In [ ]:
sales.merge(prices, on='ProductID', how='left')

ID 4 stays (`Price` is NaN). ID 5 is silently dropped.

### (c) `right` — all rows of the **right** table

In [ ]:
sales.merge(prices, on='ProductID', how='right')

### (d) `outer` — **union** of both tables

In [ ]:
sales.merge(prices, on='ProductID', how='outer')

Both ID 4 and ID 5 appear; missing values become `NaN`.

### (e) `indicator=True` — track origin of each row

In [ ]:
sales.merge(prices, on='ProductID', how='outer', indicator=True)

The `_merge` column tells you whether each row came from `left_only`, `right_only`, or `both`.

### (f) Different key names — `left_on=` / `right_on=`

In [ ]:
prices2 = prices.rename(columns={'ProductID': 'PID'})
sales.merge(prices2, left_on='ProductID', right_on='PID')

Both key columns are kept (`ProductID` and `PID`). Drop one if you don\u2019t need both.

### (g) Duplicate keys on **one** side — *one-to-many*

When a key appears multiple times on the right, the corresponding left row gets **repeated** — the row count grows.

In [ ]:
ratings = pd.DataFrame({
    'ProductID': [1, 1, 2, 2, 2, 3],
    'Reviewer':  ['A','B','A','B','C','A'],
    'Stars':     [5, 4, 3, 4, 5, 2]
})
ratings

In [ ]:
merged = sales.merge(ratings, on='ProductID')
print(f'Rows: sales={len(sales)}, ratings={len(ratings)}, merged={len(merged)}')
merged

Notice `Sold` is repeated (10 for ID 1 appears twice, 25 for ID 2 appears three times). This is normal for *one-to-many* lookups.

### (h) Duplicate keys on **both** sides — *many-to-many* (Cartesian explosion)

If both sides have duplicate keys, every left row matches every right row with the same key. The output can blow up unexpectedly.

In [ ]:
L = pd.DataFrame({'k': [1, 1, 2], 'v_l': ['a','b','c']})
R = pd.DataFrame({'k': [1, 1, 2], 'v_r': ['x','y','z']})
display(L); display(R)
out = L.merge(R, on='k')
print(f'L: {len(L)} rows, R: {len(R)} rows  ->  merged: {len(out)} rows')
out

For `k=1`: 2 rows in L \u00d7 2 rows in R = **4 rows**. For `k=2`: 1 \u00d7 1 = 1 row. Total **5 rows from 3+3 originals.** This is a very common source of bugs.

### (i) Catch unintended duplicates with `validate=`

Add `validate=` to assert your assumption about the relationship. Pandas raises `MergeError` if it doesn't hold.

In [ ]:
try:
    L.merge(R, on='k', validate='one_to_one')
except Exception as e:
    print(f'{type(e).__name__}: {e}')

Valid options: `'one_to_one'`, `'one_to_many'`, `'many_to_one'`, `'many_to_many'`. Use `validate='many_to_one'` for a typical lookup join \u2014 it will warn you if your right-side key column isn't unique.

---
# Part B — Sampling

## 6. Deterministic vs Random Sample

In [ ]:
# Deterministic: specific positions
top.iloc[[3, 18, 100], :]

In [ ]:
# Deterministic: rows matching a condition
top.loc[top['Title'].str.contains('Star', regex=False)].head()

In [ ]:
# Random: simple random sample (without replacement)
top.sample(n=5, replace=False, random_state=2026)

In [ ]:
# Random with replacement
top.sample(n=5, replace=True, random_state=2026)

---
## 7. Empirical Distribution — rolling a die

In [ ]:
die = pd.DataFrame({'Face': np.arange(1, 7)})

def empirical_hist_die(n, seed=2026):
    sample = die.sample(n=n, replace=True, random_state=seed)
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.histplot(data=sample, x='Face',
                 bins=np.arange(0.5, 6.6, 1),
                 stat='percent', discrete=True, ax=ax)
    ax.axhline(100/6, color='red', linestyle='--', label='Probability (1/6)')
    ax.legend()
    ax.set_title(f'Sample size = {n}')
    plt.show()

for n in (10, 100, 1000, 10000):
    empirical_hist_die(n)

Observe: as $n$ grows, the empirical distribution approaches the uniform probability distribution (Law of Large Numbers).

### Real data — United Airline flight delays

In [ ]:
united = pd.read_csv(f'{DATA_BASE}/united_summer2015.csv')
united.head()

In [ ]:
united['Delay'].describe()

In [ ]:
delay_bins = np.append(np.arange(-20, 301, 10), 600)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)

sns.histplot(data=united, x='Delay', bins=delay_bins, stat='percent', ax=axes[0])
axes[0].set_title('Population: all flights')
axes[0].set_xlim(-30, 200)

sample = united.sample(n=1000, replace=True, random_state=2026)
sns.histplot(data=sample, x='Delay', bins=delay_bins, stat='percent', ax=axes[1])
axes[1].set_title('Sample of 1000')
axes[1].set_xlim(-30, 200)

plt.tight_layout()
plt.show()

---
## 8. Parameter vs Statistic

In [ ]:
# Treat the full table as the ``population''
print('Parameter 1 (median delay):', np.median(united['Delay']))
print('Parameter 2 (proportion <= 2 min):', round((united['Delay'] <= 2.0).mean(), 4))
print('Parameter 3 (#flights delayed exactly 2 min):', (united['Delay'] == 2.0).sum())

In [ ]:
# Statistic: varies from sample to sample
for i in range(5):
    s = united['Delay'].sample(n=1000, replace=True)
    print(f'Sample {i+1} median = {np.median(s):.1f}')

---
## 9. Simulating a sampling distribution

In [ ]:
def random_sample_median(size):
    return np.median(united['Delay'].sample(n=size, replace=True))

# 5,000 repetitions
medians = np.array([random_sample_median(1000) for _ in range(5000)])

simulated = pd.DataFrame({'Sample Median': medians})
simulated['Sample Median'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(data=simulated, x='Sample Median',
             bins=np.arange(-0.5, 5, 1),
             stat='percent', discrete=True, ax=ax)
ax.axvline(np.median(united['Delay']), color='red', linestyle='--',
           label=f"Population median = {np.median(united['Delay'])}")
ax.legend()
ax.set_title('Empirical distribution of sample median (n=1000, 5000 reps)')
plt.show()

The **center** of this empirical distribution lands near the population parameter, and its **spread** quantifies sampling uncertainty — the entry point to *Estimation and Confidence Intervals* next time.